# 使用HuggingFace TRL微调Qwen1.5-7B模型（SFT）

## 一、指令微调

### 1. 数据集
将原始数据以Instruction(指令)-Input(输入)-Output(输出)的形式进行组织，其中Input(输入)可以为空
```
{
    "Instruction": "Add the numbers",
    "Input": "1 2 3 4 5",
    "Output": "15"
}
```

### 2. 模型微调

In [ ]:
from datasets import load_dataset
import torch
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Hugging Face model id
model_id = "./models/Qwen1.5-7B"

# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'right'
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id


# Load Dataset
train_dataset = load_dataset(
    "json", data_files="./data/sougou/train.json")['train']
test_dataset = load_dataset(
    "json", data_files="./data/sougou/test.json")['train']
print(train_dataset[0])
print(f"train size: {len(train_dataset)}, test size: {len(test_dataset)}")

# LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
    task_type="CAUSAL_LM",
)


# Using TrainingArguments
args = TrainingArguments(
    output_dir="output/sougou",  # directory to save and repository id
    num_train_epochs=3,                     # number of training epochs
    per_device_train_batch_size=8,          # batch size per device during training
    per_device_eval_batch_size=8,           # batch size per device during training
    # number of steps before performing a backward/update pass
    gradient_accumulation_steps=2,
    # use gradient checkpointing to save memory
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",               # optimizer
    save_strategy="epoch",                  # save by epoch
    evaluation_strategy="epoch",            # evaluate by rpoch
    logging_strategy="steps",               # log by step
    logging_steps=20,                       # log every 20 steps
    bf16=True,                              # use bfloat16 precision
    learning_rate=2e-4,                     # learning rate, based on QLoRA paper
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    warmup_ratio=0.1,                       # warmup ratio
    lr_scheduler_type="linear",             # use constant learning rate scheduler
    push_to_hub=False,                      # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
)

max_seq_length = 512

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    packing=False
)

# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model
trainer.save_model()

### 3. 模型测试
训练完后，会在对应的otutput目录下生成adaper模型文件，我们加载该模型，并在测试集上进行评估，脚本如下

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer


def format_instruction(text):
    string = f"""### 指令:
给定以下类别标签：['体育', '军事', '教育', '健康', '汽车']，请问下面的输入应当属于哪个类别？

### 输入:
{text}

### 输出:
"""
    return string


peft_model_id = "./output/sougou/checkpoint-750"
model = AutoModelForCausalLM.from_pretrained(peft_model_id, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained("./models/Qwen1.5-7B")


def predict(text):
    input_text = format_instruction(text=text)
    encoding = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**encoding, max_new_tokens=10, temperature=0.1,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[:, encoding.input_ids.shape[1]:]
    generated_texts = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=False)
    return generated_texts[0].split('\n')[0]


df = pd.read_csv("./data/sougou/test.csv")

true_labels, pred_labels = [], []
for i, row in df.iterrows():
    text, label = row["content"], row["label"]
    predict_label = predict(text=text[:450])
    true_labels.append(label)
    pred_labels.append(predict_label)
    print(i, label, repr(predict_label))

print(classification_report(y_true=true_labels, y_pred=pred_labels, digits=4))

## 二、对话微调
为了改善上述训练后模型的行为使用对话微调，即使用对话模板来加工训练数据。

在transformers中的tokenizer中引入了apply_chat_template方法
```
from transformers import AutoTokenizer
model_id = "./models/Qwen1.5-7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("默认模板: ")
print(tokenizer.default_chat_template)
chat = [{"role": "user", "content": "你好吗?"},
        {"role": "user", "content": "我很好。"}]
print("对话格式: ")
print(tokenizer.apply_chat_template(chat, tokenize=False))
```
```
{% for message in messages %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}
对话格式: 
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
你好吗?<|im_end|>
<|im_start|>user
我很好。<|im_end|>
```
Qwen1.5-7B模型的默认对话模板为GPT3.5模型的ChatML格式（无BOS/EOS这两个token）

### 1. 数据集
将数据加工成对话形式
```
[
    {
        "messages": [
            {
                "role": "user",
                "content": "届数比赛时间比赛地点参加国家和地区冠军亚军决赛成绩第一届1956-1957英国11美国丹麦6：1第二届1959-1960费城（美国）14美国英国5：2第三届1962-1963威尔明顿（美国）11美国英国4：3第四届1965-1966惠灵顿（新西兰）17日本美国5：2第五届1968-1969东京（日本）19日本印尼6：1第六届1971-1972东京（日本）17日本印尼6：1第七届1974-1975雅加达（印尼）14印尼日本5：2第八届1977-1978奥克兰（新西兰）16日本印尼5：2第九届1980-1981东京（日本）15日本印尼4：3第十届1983-1984吉隆坡（马来西亚）23中国印尼5：0第十一届1986雅加达（印尼）34中国印尼3：2第十二届1988吉隆坡（马来西亚）31中国韩国5：0第十三届1990东京（日本）42中国韩国3：2第十四届1992吉隆坡（马来西亚）44中国韩国3：2第十五届1994雅加达（印尼）44印尼中国3：2第十六届1996香港（中国）50印尼中国4：1第十七"
            },
            {
                "role": "assistant",
                "content": "体育"
            }
        ]
    },
    ...
]
```

### 2. 模型微调

In [ ]:
from datasets import load_dataset
import torch
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Hugging Face model id
model_id = "./models/Qwen1.5-7B"

# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'right'
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id


def formatting_prompts_func(samples):
    output_texts = []
    for i in range(len(samples['messages'])):
        text = tokenizer.apply_chat_template(
            samples['messages'][i], tokenize=False)
        output_texts.append(text)
    return output_texts


train_dataset = load_dataset(
    "json", data_files="./data/sougou/train_chat.json")['train']
test_dataset = load_dataset(
    "json", data_files="./data/sougou/test_chat.json")['train']
print(train_dataset[0])
print(f"train size: {len(train_dataset)}, test size: {len(test_dataset)}")

# LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
    task_type="CAUSAL_LM",
)

args = TrainingArguments(
    output_dir="output/sougou",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    evaluation_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    bf16=True,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="constant",
    push_to_hub=False,
    report_to="tensorboard",
)

max_seq_length = 512

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    packing=False,
    formatting_func=formatting_prompts_func
)

# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model
trainer.save_model()

### 3. 模型测试
对训练好的模型进行预测(注意generate函数的参数)

In [ ]:
from sklearn.metrics import classification_report
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

peft_model_id = "./output/sougou/checkpoint-750"
model = AutoModelForCausalLM.from_pretrained(peft_model_id, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained("/data-ai/usr/lmj/models/Qwen1.5-7B")


def predict(text):
    eos_token_id = tokenizer("<|im_end|>", add_special_tokens=False)[
        "input_ids"][0]
    encoding = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(**encoding, max_new_tokens=10, temperature=0.1,
                             do_sample=True, eos_token_id=eos_token_id, pad_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[:, encoding.input_ids.shape[1]:]
    generated_texts = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True)
    return generated_texts[0]


dataset = load_dataset(
    "json", data_files="./data/sougou/test_chat.json")['train']

true_labels, pred_labels = [], []
for i in range(len(dataset)):
    message = dataset[i]["messages"][:1]
    true_label = dataset[i]["messages"][1]["content"]
    true_labels.append(true_label)
    text = tokenizer.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True)
    pred_label = predict(text=text)
    pred_labels.append(pred_label)
    print(i, true_label, pred_label)

print(classification_report(y_true=true_labels, y_pred=pred_labels, digits=4))

设置了预测生成时的eos_token_id，这样我们就能控制大模型生成的行为了，即生成的文本只有文本分类任务的类别，这就达到了我们的目标